# Variants and process executions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LukasLiss/totem-tool/blob/main/totem_lib/docs/guide/variants.ipynb)

A process execution is one run of the process: a group of objects and every event that touches them. Executions with the same shape form a variant. This notebook splits a tiny log into process executions, groups them into variants, and compares two variants by their edit distance.

## Setup

In Google Colab, the first cell installs totem-lib from PyPI, where it is called `totem-tool`. Everywhere else it does nothing, so install totem-lib first as shown in [Installation](../installation.md).

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install --quiet totem-tool "pulp<4"

The example log comes from the totem-lib repository. Outside a clone of the repository, the next cell downloads it (11 KB).

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

LOG = Path("../../test_data/small/resource_aware_orders.json")
if not LOG.exists():
    LOG = Path("resource_aware_orders.json")
    if not LOG.exists():
        urlretrieve(
            "https://raw.githubusercontent.com/LukasLiss/totem-tool/main/"
            "totem_lib/test_data/small/resource_aware_orders.json",
            LOG,
        )

## Load the log

The log has four orders with six items, packed into three packages. Orders `o3` and `o4` share package `p3`. One worker, `w1`, picks, packs and ships everything.

The variant functions work on a DuckDB database. `import_ocel_db` reads the log into one, in memory.

In [ ]:
import polars as pl
from totem_lib.ocel import import_ocel_db

db = import_ocel_db(LOG)

pl.Config.set_tbl_rows(25)  # show every row of these small tables
db.get_events_with_objects().group_by("event_id", "activity", maintain_order=True).agg(
    objects=pl.col("obj_id").sort().str.join(", ")
)

## Process executions

`extract_process_executions` splits the log into process executions. The `extraction` argument picks the technique:

- `"leading_1hop"`: one execution per object of `leading_type`, with the objects that share an event with it
- `"leading_bfs"`: like `"leading_1hop"`, but also follows indirect links. Of each object type, it keeps only the closest objects.
- `"connected"`: one execution per group of connected objects. Two objects are connected if they share an event.
- `"resource_aware"`: like `"connected"`, but only objects of the `business_object_types` connect each other

In this log, the worker shares an event with every other object. So `"connected"` puts the whole log into one execution:

In [ ]:
from totem_lib import extract_process_executions

executions = extract_process_executions(db, extraction="connected")
executions.execution_count

With `"resource_aware"`, the worker is a resource and does not connect objects. The result has one execution per package, and the two orders that share `p3` stay together:

In [ ]:
BUSINESS_TYPES = ["order", "item", "package"]

executions = extract_process_executions(
    db, extraction="resource_aware", business_object_types=BUSINESS_TYPES
)
for case_id in sorted(executions.case_objects):
    print(f"{case_id}: objects {sorted(executions.case_objects[case_id])}")
    print(f"    events {executions.case_events[case_id]}")

The worker is not part of any execution, but an execution still has every event that touches one of its objects, such as the picks. The two shift events touch only the worker, so they belong to no execution.

## Variants

`find_variants` extracts the executions in the same way and groups the ones with the same shape into variants: the same activities, in the same order, with the same flow of objects between them. Each variant has an `id`, a `support` (its number of executions), the `case_ids` of these executions and a representative `graph`.

In [ ]:
from totem_lib import find_variants

variants = find_variants(
    db,
    extraction="resource_aware",
    business_object_types=BUSINESS_TYPES,
    verbose=False,  # no progress bars
)
for variant in variants:
    print(f"{variant.id}: support {variant.support}, executions {sorted(variant.case_ids)}")
    print("    " + " -> ".join(variant.graph.graph["sequence"]))

Orders `o1` and `o2` run the same way, so their executions share one variant. The execution with two orders is a variant of its own.

## Edit distance

`process_execution_edit_distance` compares two process executions. It returns the total cost and the list of edits that turn the first execution into the second. For a variant, it uses the representative graph. There are five kinds of edits: `add_event`, `delete_event`, `move_event` (change the activity of an event), `add_objects` and `remove_objects`.

In [ ]:
from totem_lib import process_execution_edit_distance

cost, edits = process_execution_edit_distance(variants[0], variants[1])
cost

By default, an edit costs the number of objects it touches, and objects are compared by their ids. These two executions have no objects in common, so even events with the same activity add to the cost.

`EditCosts` sets other costs. To compare only the activities, let every event edit cost 1 and every object edit cost 0:

In [ ]:
from totem_lib import EditCosts

costs = EditCosts.from_constants(
    add_event=1, delete_event=1, move_event=1, add_objects=0, remove_objects=0
)
cost, edits = process_execution_edit_distance(variants[0], variants[1], costs=costs)
print(f"Total cost: {cost}")
pl.DataFrame(edits)

The second variant has one more `place order` and one more `close order` event, one for each extra order.

## Next steps

The [API reference](../api/variants.rst) lists all functions for variants and process executions.